In [1]:
import os
print(os.getcwd())

/home/shashank-saraswat/Projects/Kidney-Disease-Classification/research


In [2]:
cd ..

/home/shashank-saraswat/Projects/Kidney-Disease-Classification


/home/shashank-saraswat/miniconda3/envs/kidney_env/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
import torch
torch = torch.load('artifacts/training/trained_model.pth')

In [18]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class EvaluationConfig:
    path_to_model: Path
    train_data_path: Path
    all_params: dict
    mlflow_uri: str
    param_image_size: list
    param_batch_size: int

In [19]:
print(os.getcwd())

/home/shashank-saraswat/Projects/Kidney-Disease-Classification


In [20]:
#configuration
from kidney_disease_prediction.constants import *
from kidney_disease_prediction.utils.common import read_yaml, create_directories, save_json

In [21]:
from pathlib import Path


class Configuration:

    def __init__(
        self,
        config_path: Path = CONFIG_FILE_PATH,
        params_path: Path = PARAMS_FILE_PATH
    ):

        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path)


    def get_evaluation_config(self) -> EvaluationConfig:

        eval_config = EvaluationConfig(

            path_to_model=Path(
                "artifacts/training/trained_model.pth"
            ),

            train_data_path=Path(
                "artifacts/data_ingestion/unzip_dir/ct-kidney-dataset-normal-cyst-tumor-and-stone"
            ),

            all_params=self.params,

            mlflow_uri=(
                "https://dagshub.com/"
                "Shashank731/"
                "Kidney-Disease-Classification.mlflow"
            ),

            param_image_size=self.params.IMAGE_SIZE,

            param_batch_size=self.params.BATCH_SIZE
        )

        return eval_config

In [25]:
from pathlib import Path
from urllib.parse import urlparse

import torch
import torch.nn as nn
import mlflow
import mlflow.pytorch

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from kidney_disease_prediction.utils.common import save_json
from kidney_disease_prediction import logger


class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config


    # 🔹 Validation Data Loader
    def valid_dataloader(self):

        transform = transforms.Compose([
            transforms.Lambda(lambda x: x.convert("RGB")),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor()
        ])

        dataset = datasets.ImageFolder(
            root=self.config.train_data_path,
            transform=transform
        )

        self.valid_loader = DataLoader(
            dataset,
            batch_size=self.config.param_batch_size,
            shuffle=False
        )

        logger.info(f"Loaded {len(dataset)} validation images")


    # 🔹 Load trained model
    def load_model(self):

        model = models.vgg16(pretrained=False)

        in_features = model.classifier[6].in_features

        model.classifier[6] = nn.Linear(
            in_features,
            self.config.all_params.CLASSES
        )

        model.load_state_dict(
            torch.load(self.config.path_to_model)
        )

        logger.info("Trained model loaded successfully")

        return model


    # 🔹 Evaluation Loop
    def evaluation(self):

        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.model = self.load_model()
        self.model.to(device)

        self.valid_dataloader()

        criterion = nn.CrossEntropyLoss()

        self.model.eval()

        running_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():

            for images, labels in self.valid_loader:

                images = images.to(device)
                labels = labels.to(device)

                outputs = self.model(images)

                loss = criterion(outputs, labels)

                running_loss += loss.item()

                _, predicted = torch.max(outputs, 1)

                total += labels.size(0)

                correct += (predicted == labels).sum().item()

        final_loss = running_loss / len(self.valid_loader)

        accuracy = 100 * correct / total

        self.score = {
            "loss": final_loss,
            "accuracy": accuracy
        }

        logger.info(
            f"Validation Loss: {final_loss:.4f}, "
            f"Accuracy: {accuracy:.2f}%"
        )

        self.save_score()


    # 🔹 Save scores
    def save_score(self):

        save_json(
            path=Path("scores.json"),
            data=self.score
        )


    # 🔹 MLflow Logging
    def log_into_mlflow(self):

        mlflow.set_tracking_uri(
            self.config.mlflow_uri
        )

        tracking_url_type_store = urlparse(
            mlflow.get_tracking_uri()
        ).scheme

        with mlflow.start_run():

            mlflow.log_params(
                dict(self.config.all_params)
            )

            mlflow.log_metrics(
                self.score
            )

            if tracking_url_type_store != "file":

                mlflow.pytorch.log_model(
                    self.model,
                    "model",
                    registered_model_name="VGG16Model"
                )

            else:

                mlflow.pytorch.log_model(
                    self.model,
                    "model"
                )

        logger.info("MLflow logging completed")

In [23]:
STAGE_NAME = "Model Evaluation Stage"


class EvaluationPipeline:

    def __init__(self):
        pass

    def main(self):

        config = Configuration()

        eval_config = config.get_evaluation_config()

        evaluation = Evaluation(config=eval_config)

        logger.info("Starting model evaluation")

        evaluation.evaluation()

        evaluation.log_into_mlflow()

        logger.info("Model evaluation completed")

In [26]:
STAGE_NAME = "Model Evaluation Stage"

logger.info(f">>>>>> stage {STAGE_NAME} started <<<<<<")

if __name__ == "__main__":

    try:

        evaluation_pipeline = EvaluationPipeline()

        evaluation_pipeline.main()

        logger.info(
            f">>>>>> stage {STAGE_NAME} completed <<<<<<\n\n"
        )

    except Exception as e:

        logger.exception(e)

        raise e

[2026-05-06 16:29:31,661: INFO: 383338996 : >>>>>> stage Model Evaluation Stage started <<<<<<]
[2026-05-06 16:29:31,663: INFO: common : yaml file: config/config.yaml loaded successfully]
[2026-05-06 16:29:31,665: INFO: common : yaml file: params.yaml loaded successfully]
[2026-05-06 16:29:31,665: INFO: 2294002535 : Starting model evaluation]


/home/shashank-saraswat/miniconda3/envs/kidney_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/shashank-saraswat/miniconda3/envs/kidney_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


[2026-05-06 16:29:32,950: INFO: 1723083960 : Trained model loaded successfully]
[2026-05-06 16:29:33,092: INFO: 1723083960 : Loaded 12446 validation images]
[2026-05-06 16:32:43,563: INFO: 1723083960 : Validation Loss: 1.7028, Accuracy: 62.78%]
[2026-05-06 16:32:43,565: INFO: common : json file saved at: scores.json]
[2026-05-06 16:32:44,509: ERROR: 383338996 : API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: '']
Traceback (most recent call last):
  File "/tmp/ipykernel_10571/383338996.py", line 11, in <module>
    evaluation_pipeline.main()
  File "/tmp/ipykernel_10571/2294002535.py", line 21, in main
    evaluation.log_into_mlflow()
  File "/tmp/ipykernel_10571/1723083960.py", line 141, in log_into_mlflow
    with mlflow.start_run():
  File "/home/shashank-saraswat/miniconda3/envs/kidney_env/lib/python3.10/site-packages/mlflow/tracking/fluent.py", line 350, in start_run
    active_run_obj = client.create_run(
  File "/home/shashank

MlflowException: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''